### Locomotion-Transformer  
### How to  
1. Gazebo を起動  
$ __NV_PRIME_RENDER_OFFLOAD=1 __GLX_VENDOR_LIBRARY_NAME=nvidia ros2 launch mini_pupper_simulation bringup.launch.py launch_twist_converter:=False  


In [ ]:
import os
import sys
# torchをインポートする前に設定
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from gazebo_env import MiniPupperEnv,MAX_ACTION_RAD
from stable_baselines3 import PPO
import rclpy
import torch

import torch as th
import torch.nn as nn
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor


In [ ]:
rclpy.init()
env = MiniPupperEnv()

cont_f=True

if MAX_ACTION_RAD==1.0:
    use_sde=True
    sde_sample_freq=16
    out_dir="outs-10"
else:
    use_sde=True       # add by nishi 2026.8.3
    sde_sample_freq=4   # add by nishi 2026.8.3
    out_dir="outs-05"
    

if not cont_f:
    reset_num_timesteps=True
else:
    reset_num_timesteps=False

In [ ]:
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
#from lerobot.datasets.lerobot_dataset import LeRobotDataset

# =====================================================================
# 1. 時系列対応版 MiniPupperLocomotionTransformer の定義
# =====================================================================
class MiniPupperLocomotionTransformer(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256, history_len=6):
        # 期待する入力形状: [バッチサイズ, history_len, 31]
        super().__init__(observation_space, features_dim)
        
        self.history_len = history_len
        self.num_legs = 4
        self.features_per_leg = 6  
        self.token_dim = 64      
        
        # --- 各入力パーツをトークンに変換する層 ---
        self.cmd_embed = nn.Linear(3, self.token_dim)      
        self.leg_embed = nn.Linear(self.features_per_leg, self.token_dim)
        self.quat_embed = nn.Linear(4, self.token_dim)     
        
        # --- 【時間軸】位置エンコーディングのパラメータ ---
        self.temporal_embedding = nn.Parameter(th.randn(self.history_len, 1, self.token_dim))
        
        # --- Transformerエンコーダー ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.token_dim, 
            nhead=4,              
            dim_feedforward=128,  
            batch_first=True,
            activation="relu"
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # --- 最終出力をまとめる層 ---
        total_tokens = self.history_len * (1 + self.num_legs + 1)  # 6ステップ × 6トークン = 36トークン
        self.output_layer = nn.Sequential(
            nn.Linear(total_tokens * self.token_dim, features_dim),
            nn.ReLU()
        )

    def forward(self, observations):
        batch_size = observations.shape[0]
        all_temporal_tokens = []
        
        # 履歴の各ステップ（時間軸）ごとに処理
        for t in range(self.history_len):
            step_obs = observations[:, t, :]  # [batch_size, 31]
            
            # 31次元のスライス (cmd_vel, joint_pos, joint_vel, quat)
            cmd_vel_data = step_obs[:, 0:3]
            joint_pos    = step_obs[:, 3:15]
            joint_vel    = step_obs[:, 15:27]
            quat_data    = step_obs[:, 27:31]
            
            # 各パーツのトークン化
            cmd_token = self.cmd_embed(cmd_vel_data).unsqueeze(1)
            
            leg_pos_split = joint_pos.view(batch_size, self.num_legs, 3)
            leg_vel_split = joint_vel.view(batch_size, self.num_legs, 3)
            legs_combined = th.cat([leg_pos_split, leg_vel_split], dim=2)
            
            leg_tokens = self.leg_embed(legs_combined)
            quat_token = self.quat_embed(quat_data).unsqueeze(1)
            
            # 空間トークン結合 + 時間エンコーディングの加算
            spatial_tokens = th.cat([cmd_token, leg_tokens, quat_token], dim=1)
            spatial_tokens = spatial_tokens + self.temporal_embedding[t]
            
            all_temporal_tokens.append(spatial_tokens)
            
        # 全時間軸・空間軸のトークンをフラットに結合
        tokens = th.cat(all_temporal_tokens, dim=1)
        
        transformer_out = self.transformer(tokens)  
        flat_out = transformer_out.view(batch_size, -1)
        return self.output_layer(flat_out)

In [ ]:
# 1. デバイスの定義 (すでに定義済みの場合は不要)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = "cpu"

# ネットワーク構造のカスタマイズ (おすすめ設定)
policy_kwargs_old = dict(
    net_arch=dict(
        pi=[256, 256],  # 政策（Policy）ネットワーク: 256×2層
        vf=[256, 256]   # 価値（Value）ネットワーク: 256×2層
    )
)

# 現在のMlpPolicyの指定をベースに、policy_kwargsを追記します
policy_kwargs_old2 = dict(
    features_extractor_class=MiniPupperLocomotionTransformer,
    features_extractor_kwargs=dict(features_dim=256),
    # 必要に応じて、Transformerの後のActor/Criticそれぞれの全結合層(頭)の厚みを指定
    #net_arch=dict(pi=[128, 128], vf=[128, 128]) 
    net_arch=dict(pi=[256, 256], vf=[256, 256]) 
)

# ポリシー設定にカスタム特徴抽出器を登録
policy_kwargs = dict(
    features_extractor_class=MiniPupperLocomotionTransformer,
    features_extractor_kwargs=dict(features_dim=256, history_len=6),
    # 必要に応じて、Transformerの後のActor/Criticそれぞれの全結合層(頭)の厚みを指定
    #net_arch=dict(pi=[128, 128], vf=[128, 128]) 
    net_arch=dict(pi=[256, 256], vf=[256, 256]) 
)

if not cont_f:
    if False:
        model = PPO(
            policy="MlpPolicy",
            env=env,
            policy_kwargs=policy_kwargs,
            learning_rate=3e-4,
            verbose=1,
            device="cuda",
            use_sde=use_sde,       # add by nishi 2026.8.3
            sde_sample_freq=sde_sample_freq,   # add by nishi 2026.8.3
        )

    if True:
        model = PPO(
            policy="MlpPolicy",           # "MlpPolicy" は、内部で ActorCriticPolicy をコールする。
            env=env,
            policy_kwargs=policy_kwargs,
            learning_rate=2e-4,        # Transformer向けに少しだけ慎重に設定（2e-4などでも可）
            n_steps=4096,              
            batch_size=128,            
            n_epochs=10,               
            gamma=0.99,                
            gae_lambda=0.95,           
            clip_range=0.2,            
            ent_coef=0.05,             # 積極的な探索設定を維持
            verbose=1,
            device=device,
            #use_sde=True,                # SDEとの相性も抜群です
            #sde_sample_freq=4,   
            use_sde=use_sde,       # add by nishi 2026.8.3
            sde_sample_freq=sde_sample_freq,   # add by nishi 2026.8.3
        )
   
    if False:
        model = PPO(
            ActorCriticPolicy,           # "MlpPolicy" からクラスオブジェクトに変更
            env,
            policy_kwargs=policy_kwargs,
            learning_rate=2e-4,        # Transformer向けに少しだけ慎重に設定（2e-4などでも可）
            n_steps=4096,              
            batch_size=128,            
            n_epochs=10,               
            gamma=0.99,                
            gae_lambda=0.95,           
            clip_range=0.2,            
            ent_coef=0.05,             # 積極的な探索設定を維持
            verbose=1,
            device=device,
            #use_sde=True,                # SDEとの相性も抜群です
            #sde_sample_freq=4,   
            use_sde=use_sde,       # add by nishi 2026.8.3
            sde_sample_freq=sde_sample_freq,   # add by nishi 2026.8.3
        )
else:
    # policy_kwargs は不要（保存されたファイルから自動で読み込まれます）
    model = PPO.load(
        os.path.join(out_dir, "ppo_minipupper_test_3"),
        #"outs/ppo_minipupper_test_1",
        env=env,
        device=device
    )

In [ ]:
model.learn(
    #total_timesteps=500000,
    #total_timesteps=1000,
    #total_timesteps=3000
    #total_timesteps=10000,
    #total_timesteps=50000,
    total_timesteps=100000,
    #total_timesteps=200000,
    #total_timesteps=300000,
    reset_num_timesteps=reset_num_timesteps
)


In [ ]:
model.save(os.path.join(out_dir, "ppo_minipupper_test_latest"))